# __FOLDER__ — one Kaggle account

1. **Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **On** (phone-verified account).
2. **Code:** either push the repository to GitHub (private repo: add a Kaggle secret `GITHUB_TOKEN`), **or** upload the folder `FMIA_Experiments/__FOLDER__` as a private Kaggle *Dataset* and add it as an input. Nothing outside this folder is needed.
3. First run with `SMOKE = True` (20-40 min) and check that `results___FOLDER___smoke/STATUS.md` says COMPLETE. Then set `SMOKE = False`.
4. **Save Version → Save & Run All (Commit).** The run stops itself before the 12-hour limit.
5. If `STATUS.md` says *NOT FINISHED*: **Add Input → Your Work → Notebooks →** this notebook (latest version), then **Save Version** again. It resumes where it stopped.

Results: `results___FOLDER__/summary/REPORT.md` (+ CSVs and figures). Full instructions: `README.md` in the folder.

In [ ]:
# ============================ EDIT ============================
SMOKE   = False     # True = quick pipeline test (tiny data, 2 epochs)
HOURS   = 11.0      # session budget; Kaggle stops a session at 12 h
PARTIAL = False     # True = also attack/report with the models finished so far
REPO    = "https://github.com/NCRL-IIITB/Evasion_MIA.git"
BRANCH  = "main"
# ===============================================================
FOLDER  = "__FOLDER__"

In [ ]:
# Get the code: an attached Kaggle dataset containing the folder, else git clone
import glob, os, subprocess, sys
hits = [os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{FOLDER}/run.py", recursive=True)]
if hits:
    WORK = "/tmp/" + FOLDER
    subprocess.run(["rm", "-rf", WORK]); subprocess.run(["cp", "-r", hits[0], WORK], check=True)
    print("code from attached dataset:", hits[0])
else:
    url = REPO
    try:   # private repository: Kaggle secret GITHUB_TOKEN (Add-ons -> Secrets)
        from kaggle_secrets import UserSecretsClient
        url = REPO.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
    except Exception:
        pass
    CODE = "/tmp/repo"
    if not os.path.isdir(CODE + "/.git"):
        r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, CODE], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("git clone failed:\n" + r.stderr.replace(url, REPO))
    print(subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%h %s (%ci)"], capture_output=True, text=True).stdout)
    WORK = f"{CODE}/FMIA_Experiments/{FOLDER}"
assert os.path.exists(WORK + "/run.py"), WORK
if "medical" in FOLDER:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "medmnist"], check=False)

In [ ]:
# Run (restores the attached previous output, trains, probes, attacks, writes the report)
cmd = [sys.executable, "-u", WORK + "/run.py", "--hours", str(HOURS)]
if SMOKE:   cmd.append("--smoke")
if PARTIAL: cmd.append("--partial")
print(" ".join(cmd))
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=WORK)
for line in p.stdout:
    print(line, end="")
p.wait(); print("exit code", p.returncode)

In [ ]:
res = f"/kaggle/working/results_{FOLDER}" + ("_smoke" if SMOKE else "")
print(open(res + "/STATUS.md").read() if os.path.exists(res + "/STATUS.md") else "no STATUS.md")
from IPython.display import Markdown, display
if os.path.exists(res + "/summary/REPORT.md"):
    display(Markdown(open(res + "/summary/REPORT.md").read()[:60000]))